## Google Trends Data Collection — Search Interest Proxy

Pulls weekly search interest for 15 outdoor-apparel keywords across three 
weather-driven categories (rain, cold, warm), using Google Trends as a proxy 
for consumer demand. Search interest is grouped into separate batches per 
category rather than compared directly against each other.

In [1]:
from pytrends.request import TrendReq
import pandas as pd
import time

### Define keyword batches and pull search interest

Google Trends limits requests to 5 keywords at a time, and normalizes each 
request's scores independently — so a score of 50 in one batch isn't directly 
comparable to a score of 50 in another. 

To work around this, keywords are grouped 
into three batches by the weather condition most likely to drive them (rain, cold, 
warm), rather than mixed together. This means categories are later compared by 
**correlation strength against their own weather driver**, not by raw search score.

A 30-second pause between calls reduces the risk of Google's rate limiting.

In [2]:
pytrends = TrendReq(hl='en-CA', tz=480)

rain_batch = ["rain jacket", "waterproof boots", "rain pants", "umbrella", "rubber boots"]
cold_batch = ["insulated jacket", "base layer", "winter gloves", "wool socks", "beanie"]
warm_batch = ["hiking boots", "sandals", "light jacket", "sun hat", "shorts"]

batches = {
    "rain": rain_batch,
    "cold": cold_batch,
    "warm": warm_batch
}

results = {}

for label, kw_list in batches.items():
    pytrends.build_payload(kw_list=kw_list, timeframe='2022-01-01 2025-12-31', geo='CA-BC')
    df = pytrends.interest_over_time()
    df = df.reset_index()
    df.columns = df.columns.str.replace(' ', '_')
    results[label] = df
    time.sleep(30)  # pause between calls to reduce rate-limit risk

rain_df = results["rain"]
cold_df = results["cold"]
warm_df = results["warm"]

### Export raw search interest data

Saved separately per category, ready for import into MySQL alongside the weather 
data for cleaning, weekly alignment, and correlation analysis.

In [ ]:
rain_df.to_csv("trends_rain.csv", index=False)
cold_df.to_csv("trends_cold.csv", index=False)
warm_df.to_csv("trends_warm.csv", index=False)